# Suggesting the Expense Type for Department for Transport spend lines

**Patrick Gilliland**

**Question.** Can a model trained on the Department for Transport's past ledger lines suggest the right Expense Type for a new line over £25,000, and what share of lines could it code confidently enough to leave the rest to a person?

**Task 1: the problem and the dataset.** Every payment line a finance team posts carries an account code, which in this data is the `Expense Type`. The problem is to suggest that code for a new line from the other fields on the line, such as the supplier and the item text. In accounting terms this is general ledger account classification: assigning an account code to each posting line. The task is supervised learning (classification), and it is multi-class, because `Expense Type` holds many labels. The dataset is [Spend over £25,000 in the Department for Transport and its arms-length bodies](https://www.data.gov.uk/dataset/804be0c6-ae01-4899-bbb9-2317dff1a4f9/financial-transactions-data-dft), which the Department for Transport (DfT) publishes on data.gov.uk as one CSV file a month. I use the 48 months from April 2022 to March 2026. `build_dataset.py` stacks the original monthly files in `dft_originals/` into `dft_spend_over_25k_2022-04_to_2026-03.csv`, beside this notebook, without changing any cell. Contains public sector information licensed under the Open Government Licence v3.0.

**Report.** The Machine Learning Analysis Report is one PDF under the two names the brief uses: `module_summary.pdf`, and an identical copy named `Machine_Learning_Analysis_Report.pdf`.

## Data preparation

### Setup

In [1]:
import hashlib
import os
import platform
from pathlib import Path

import matplotlib
import numpy as np
import pandas as pd
import sklearn

# I show every column and whole cell text, up to 500 rows so the anomaly
# listings appear in full, and figures with two decimal places and thousands
# separators, so the wide tables below read without scientific notation.
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 500)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", "{:,.2f}".format)

# Every step below that draws at random takes this seed, so a rerun gives the
# same numbers.
SEED = 42

print(f"Python {platform.python_version()}")
print(f"NumPy {np.__version__}")
print(f"pandas {pd.__version__}")
print(f"scikit-learn {sklearn.__version__}")
print(f"Matplotlib {matplotlib.__version__}")
print(f"Random seed: {SEED}")
print(f"CPU cores: {os.cpu_count()}")

Python 3.12.3
NumPy 2.5.3
pandas 3.0.6
scikit-learn 1.9.1
Matplotlib 3.11.2
Random seed: 42
CPU cores: 24


**Hardware.** I run everything on the CPU, with no GPU. The models are scikit-learn's, and the whole file fits in memory, so I load it in one go. Where a scikit-learn step can use every core, it gets `n_jobs=-1`.

### Task 2: Load and inspect the dataset

In [2]:
DATA = Path("dft_spend_over_25k_2022-04_to_2026-03.csv")
EXPECTED_SHA256 = (
    "3d9d7f911ef00f9394c07077ebb1ec161efbeb7f3073824c801a662aac212bd4"
)

if not DATA.exists():
    raise FileNotFoundError(
        f"{DATA} is not beside this notebook. Build it from the 48 original "
        "monthly files in dft_originals/ with: python build_dataset.py"
    )
digest = hashlib.sha256(DATA.read_bytes()).hexdigest()
print(f"File: {DATA}, {DATA.stat().st_size:,} bytes")
print(f"SHA-256: {digest}")
# Every figure below comes from this exact file, so I stop rather than
# analyse a different one.
if digest != EXPECTED_SHA256:
    raise ValueError(
        f"{DATA} is not the file build_dataset.py writes from the 48 original "
        "files. Rebuild it with: python build_dataset.py"
    )
print(
    "It matches the file build_dataset.py writes from the 48 original files."
)

# I read every cell as the text DfT published. The amounts carry a pound sign
# and commas, and by default pandas would turn text such as "#N/A" into a
# blank, so nothing is converted on load.
df = pd.read_csv(DATA, dtype=str, keep_default_na=False)
print(f"Loaded: {len(df):,} rows, {df.shape[1]} columns")

File: dft_spend_over_25k_2022-04_to_2026-03.csv, 25,396,696 bytes
SHA-256: 3d9d7f911ef00f9394c07077ebb1ec161efbeb7f3073824c801a662aac212bd4
It matches the file build_dataset.py writes from the 48 original files.
Loaded: 126,346 rows, 11 columns


In [3]:
df.head()

,Department Family,Entity,Date,Expense Type,Expense Area,Supplier,Transaction No,Item Text,£,Postal Code,source_file
0,Department for Transport,Department for Transport,05/04/2022,Grt Aid to NDPBs,DG Rail Strategy and Services Group,NETWORK RAIL ISTRE (CAPITAL GRANTS),2000000612,NR GRANT APRIL 2 SR 9000946295,"£418,408,000.00",NW1 2DN,dft_2022-04.csv
1,Department for Transport,Department for Transport,29/04/2022,Grt Aid to NDPBs,"DG Roads, Places and Environment",National Highways Limited,2000003150,Grant in Aid NH for May 2022,"£298,000,000.00",B1 1RN,dft_2022-04.csv
2,Department for Transport,Department for Transport,29/04/2022,CurrGrtOth to NDPBs,DG Rail Strategy and Services Group,NETWORK RAIL ISTRE (CAPITAL GRANTS),2000003127,NR Grant - DUE 03/05/2022 - SR 9000953144,"£151,086,319.00",NW1 2DN,dft_2022-04.csv
3,Department for Transport,High Speed Two (HS2) Limited,07/04/2022,AUC - Phase 1,000000 AUC - Phase 1,BALFOUR BEATTY VINCI JV - HS2 (N2),#,MWCC Stage One TP,"£135,660,641.62",#,dft_2022-04.csv
4,Department for Transport,Department for Transport,05/04/2022,Cap Grt NDPBs & CG,DG Rail Strategy and Services Group,NETWORK RAIL ISTRE (CAPITAL GRANTS),2000000612,NR GRANT APRIL 2 SR 9000946295,"£117,640,000.00",NW1 2DN,dft_2022-04.csv


In [4]:
published = [name for name in df.columns if name != "source_file"]
stripped = df.apply(lambda column: column.str.strip())
blank_row = stripped[published].eq("").all(axis=1)
blank_files = ", ".join(df.loc[blank_row, "source_file"].unique())
print(
    f"Rows: {len(df):,}, of which {blank_row.sum():,} have every published "
    f"cell blank, all from {blank_files}"
)

# The publisher marks a missing value with "#", and a few cells hold the
# spreadsheet error "#N/A", so I count those beside the truly blank cells.
column_check = pd.DataFrame(
    {
        "dtype": df.dtypes.astype(str),
        "blank": stripped.eq("").sum(),
        '"#"': stripped.eq("#").sum(),
        '"#N/A" or "N/A"': stripped.isin(["#N/A", "N/A"]).sum(),
        "spaces at either end": df.ne(stripped).sum(),
        "distinct values": df.where(stripped.ne("")).nunique(),
    }
)
column_check

Rows: 126,346, of which 657 have every published cell blank, all from dft_2022-05.csv


,dtype,blank,"""#""","""#N/A"" or ""N/A""",spaces at either end,distinct values
Department Family,str,657,0,0,137,3
Entity,str,657,0,0,1260,15
Date,str,657,0,0,1,1387
Expense Type,str,657,0,0,666,1054
Expense Area,str,657,0,4,756,976
Supplier,str,657,0,0,381,4015
Transaction No,str,657,10192,2,0,97175
Item Text,str,657,16139,57,1977,20231
£,str,657,0,0,50587,104351
Postal Code,str,657,16978,0,210,2791


In [5]:
def parse_fields(frame, amount="£", date="Date", source="source_file"):
    """Return a copy of frame with the amount, date and month as typed columns.

    Three columns are added. amount is the amount in pounds as a float,
    read from text such as " £1,234.50 ". date is the payment date, read as
    day/month/year, or as day.month.year where the file uses dots. month is
    the month of the file the row came from, "YYYY-MM", taken from the
    source file's name. A value that cannot be read is left blank (NaN or
    NaT). frame is not changed.

    Parameters
    ----------
    frame : pandas.DataFrame
        The rows as published, every column text.
    amount, date, source : str
        The names of the amount, date and source file columns.

    Returns
    -------
    pandas.DataFrame
        frame's columns unchanged, then amount, date and month.
    """
    out = frame.copy()
    money = frame[amount].str.strip().str.replace("£", "", regex=False)
    money = money.str.replace(",", "", regex=False)
    out["amount"] = pd.to_numeric(money, errors="coerce").astype(float)
    text = frame[date].str.strip()
    slashes = pd.to_datetime(text, format="%d/%m/%Y", errors="coerce")
    dots = pd.to_datetime(text, format="%d.%m.%Y", errors="coerce")
    out["date"] = slashes.fillna(dots)
    out["month"] = frame[source].str.extract(r"(\d{4}-\d{2})", expand=False)
    return out

In [6]:
df = parse_fields(df)
content = ~blank_row
written = df.loc[content, "Date"]
slash_form = written.str.fullmatch(r"\d{2}/\d{2}/\d{4}")
dot_form = written.str.fullmatch(r"\d{2}\.\d{2}\.\d{4}")
spaced = written.ne(written.str.strip())
print(
    f"Amounts read: {df.loc[content, 'amount'].notna().sum():,} of "
    f"{content.sum():,} lines with content"
)
print(
    f"Dates read: {df.loc[content, 'date'].notna().sum():,} of "
    f"{content.sum():,}; {slash_form.sum():,} written dd/mm/yyyy, "
    f"{dot_form.sum():,} written dd.mm.yyyy, {spaced.sum():,} with spaces "
    "round it"
)
months = df["month"]
print(f"Months: {months.nunique()}, {months.min()} to {months.max()}")
print(f"Published total: £{df['amount'].sum():,.2f}")
print(
    f"Smallest line: £{df['amount'].min():,.2f}; "
    f"largest: £{df['amount'].max():,.2f}"
)
df[["amount", "date", "month"]].dtypes

Amounts read: 125,689 of 125,689 lines with content
Dates read: 125,689 of 125,689; 125,660 written dd/mm/yyyy, 28 written dd.mm.yyyy, 1 with spaces round it
Months: 48, 2022-04 to 2026-03
Published total: £177,111,208,418.51
Smallest line: £25,000.00; largest: £786,575,788.00


amount           float64
date      datetime64[us]
month                str
dtype: object

**What loaded.** The file loads as 126,346 rows and 11 columns: the ten columns DfT publishes and `source_file`, which names the monthly file each row came from. I read every cell as text, so every column loads as text. `parse_fields` then adds the amount as a number, the date as a date and the file's month. Every one of the 125,689 lines with content has an amount and a date I can read, across 48 months.

**Obvious data quality issues.**

- 657 rows are blank in every published column, all from the May 2022 file. The build keeps them, so the combined file holds every row DfT published.
- The amount is text with a pound sign and commas, and 50,587 amount cells have spaces round them. Every amount reads, from £25,000.00 to £786,575,788.00, so no line is a credit.
- The date is text too. 125,660 lines write it dd/mm/yyyy, 28 use dots and 1 has spaces round it. `parse_fields` reads all three forms.
- The publisher writes "#" for a missing value: in 10,192 transaction numbers, 16,139 item texts and 16,978 postcodes. The spreadsheet error "#N/A" or "N/A" sits in 4 Expense Area cells, 2 transaction numbers and 57 item texts. I read the file with `keep_default_na=False`, so pandas cannot turn any of these into a blank without my knowing.
- Names are not always written one way. Department Family has 3 spellings for one department, Entity has 15 values, with spaces round 1,260 of its cells, and Expense Type has 1,054 labels, some of which differ only in case, spacing or punctuation.

I list each of these, and the other anomalies I found, with its count and value in the anomaly list below, before anything is cleaned.

### The anomaly list

Before anything is cleaned, I list every anomaly I could find in the 48 months, kind by kind. For each kind the anomaly list gives the lines it touches, their value, and the published total with and without them. Each kind also has my proposed treatment, flag, keep or exclude, which waits for a ruling. Nothing is removed here: Task 3 applies the rulings, keeps every raw row and gives any excluded line its reason. First come the listings the list draws on: the original files, the time-ordered split, the label variants, the truncation pairs and National Highways' newer "TA Cost AUC" labels.

In [7]:
def file_anomalies(manifest, schema, usual_encoding="cp1252"):
    """Return the original files whose encoding or header is not the usual one.

    A file is listed when its encoding is not usual_encoding, or when a
    column in its header is named differently from schema at the same
    position. manifest is not changed.

    Parameters
    ----------
    manifest : pandas.DataFrame
        One row per original file: file, month, encoding, rows (text, as
        the manifest stores it) and header, a tuple of its trimmed column
        names.
    schema : list of str
        The column names the build maps every header to.
    usual_encoding : str, default "cp1252"
        The encoding most of the files use.

    Returns
    -------
    pandas.DataFrame
        Indexed by file: month, encoding, rows as a number, and the columns
        named differently, each as '"published name" for "schema name"',
        or "" when the header is the usual one.
    """
    renamed = manifest["header"].map(
        lambda header: "; ".join(
            f'"{name}" for "{wanted}"'
            for name, wanted in zip(header, schema)
            if name != wanted
        )
    )
    out = manifest[["file", "month", "encoding"]].assign(
        rows=manifest["rows"].astype(int),
        **{"columns named differently": renamed},
    )
    odd = (out["encoding"] != usual_encoding) | (renamed != "")
    return out[odd].set_index("file")

In [8]:
# The build reads each original with the encoding its manifest gives, and
# maps each header to one set of column names. I read both off the
# originals here, so the files that needed either are shown, not assumed.
ORIGINALS = Path("dft_originals")
manifest = pd.read_csv(ORIGINALS / "MANIFEST.csv", dtype=str)
manifest["header"] = [
    tuple(
        name.strip()
        for name in pd.read_csv(
            ORIGINALS / row.file, encoding=row.encoding, nrows=0
        ).columns
    )
    for row in manifest.itertuples()
]
file_check = file_anomalies(manifest, schema=published)
cp850_files = list(file_check.index[file_check["encoding"] == "cp850"])
renamed = file_check["columns named differently"] != ""
drift_files = list(file_check.index[renamed])
print(
    f"Original files: {len(manifest)}; not cp1252: "
    f"{(file_check['encoding'] != 'cp1252').sum()}; with a column named "
    f"differently: {len(drift_files)}"
)
file_check

Original files: 48; not cp1252: 5; with a column named differently: 5


,month,encoding,rows,columns named differently
file,,,,
dft_2022-05.csv,2022-05,cp1252,3271,"""Expense Area rev"" for ""Expense Area"""
dft_2022-08.csv,2022-08,cp1252,2232,"""Expense Area rev"" for ""Expense Area"""
dft_2023-04.csv,2023-04,cp850,2879,"""Departmental Family"" for ""Department Family"""
dft_2023-07.csv,2023-07,cp850,2842,
dft_2023-08.csv,2023-08,cp850,2596,
dft_2024-03.csv,2024-03,cp1252,3515,"""Transaction Number"" for ""Transaction No"""
dft_2025-01.csv,2025-01,utf-8-sig,2556,
dft_2025-02.csv,2025-02,utf-8-sig,2586,
dft_2025-07.csv,2025-07,cp1252,2586,"""Departmental Family"" for ""Department Family"""


**The original files.** Of the 48 original files, 5 are not cp1252. January and February 2025 are UTF-8 and start with a byte-order mark. April, July and August 2023 store the pound sign as the single byte 0x9C, which cp850 reads as "£", so the build reads those 3 files as cp850. Separately, 5 files name a column differently, such as "Expense Area rev" in May and August 2022, and the build maps each one to the usual name. Neither changes a published value, so I propose to keep the lines from these files as they are.

In [9]:
def assign_split(
    frame, month="month", training_end="2025-06", validation_end="2025-09"
):
    """Return the time-ordered split each row falls in, from its month.

    A row is in training up to and including training_end, in validation
    up to and including validation_end, and in test after that. Months are
    "YYYY-MM" text, which sorts in time order. frame is not changed.

    Parameters
    ----------
    frame : pandas.DataFrame
        Rows with a month column.
    month : str, default "month"
        The month column.
    training_end, validation_end : str
        The last month of training and of validation.

    Returns
    -------
    pandas.Series
        "training", "validation" or "test" on frame's index, named split.
    """
    months = frame[month]
    split = np.where(
        months <= training_end,
        "training",
        np.where(months <= validation_end, "validation", "test"),
    )
    return pd.Series(split, index=frame.index, name="split")

In [10]:
df["split"] = assign_split(df)
lines = df[content]
for name in ("training", "validation", "test"):
    in_split = lines.loc[lines["split"] == name, "month"]
    print(
        f"{name}: {in_split.min()} to {in_split.max()}, "
        f"{in_split.nunique()} months, {len(in_split):,} lines"
    )

training: 2022-04 to 2025-06, 39 months, 102,081 lines
validation: 2025-07 to 2025-09, 3 months, 7,291 lines
test: 2025-10 to 2026-03, 6 months, 16,317 lines


In [11]:
def label_variants(frame, column="Expense Type"):
    """Return every label that is another spelling of a commoner label.

    Two labels are spellings of one label when they match once case and
    every character that is not a letter or a digit are ignored. In each
    group the spelling with the most lines is the commonest, and a tie goes
    to the spelling that sorts first. Every other spelling is listed with
    the kind of difference, the first of these that applies:

    - spaces: it matches once runs of spaces are closed up and the ends
      trimmed;
    - case: it then matches ignoring case;
    - dash: it then matches once each en dash, em dash or "?" is read as a
      hyphen;
    - separator: any other difference, such as "/" for "-".

    Blank labels are left out. frame is not changed.

    Parameters
    ----------
    frame : pandas.DataFrame
        One row per line.
    column : str, default "Expense Type"
        The label column.

    Returns
    -------
    pandas.DataFrame
        One row per spelling that is not the commonest: label, commonest
        spelling, kind, lines and lines in commonest. Sorted by kind in the
        order above, then by lines, most first.
    """
    labels = frame[column]
    counts = labels[labels.str.strip() != ""].value_counts()
    table = pd.DataFrame({"label": counts.index, "lines": counts.to_numpy()})
    table = table.sort_values(["lines", "label"], ascending=[False, True])
    table["key"] = table["label"].str.casefold()
    table["key"] = table["key"].str.replace(r"[^a-z0-9]", "", regex=True)
    group = table.groupby("key")
    table["commonest spelling"] = group["label"].transform("first")
    table["lines in commonest"] = group["lines"].transform("first")
    out = table[table["label"] != table["commonest spelling"]].copy()

    label = out["label"]
    best = out["commonest spelling"]
    as_hyphen = str.maketrans({chr(0x2013): "-", chr(0x2014): "-", "?": "-"})
    closed = label.str.replace(r"\s+", " ", regex=True).str.strip()
    closed_best = best.str.replace(r"\s+", " ", regex=True).str.strip()
    hyphened = closed.str.translate(as_hyphen).str.casefold()
    hyphened_best = closed_best.str.translate(as_hyphen).str.casefold()
    kinds = ["spaces", "case", "dash"]
    out["kind"] = np.select(
        [
            closed == closed_best,
            closed.str.casefold() == closed_best.str.casefold(),
            hyphened == hyphened_best,
        ],
        kinds,
        default="separator",
    )
    out["order"] = out["kind"].map({kind: i for i, kind in enumerate(kinds)})
    out["order"] = out["order"].fillna(len(kinds))
    out = out.sort_values(
        ["order", "lines", "label"], ascending=[True, False, True]
    )
    columns = ["label", "commonest spelling", "kind", "lines"]
    return out[columns + ["lines in commonest"]].reset_index(drop=True)

In [12]:
variants = label_variants(df)
# I rank each spelling by its lines in the training months, ties in label
# order, so a pair with both spellings among the commonest types shows up.
in_training = lines.loc[lines["split"] == "training", "Expense Type"]
ranked = in_training.value_counts().sort_index()
ranked = ranked.sort_values(ascending=False, kind="stable")
rank = pd.Series(range(1, len(ranked) + 1), index=ranked.index)
variants["training rank"] = variants["label"].map(rank).astype("Int64")
variants["training rank (commonest)"] = (
    variants["commonest spelling"].map(rank).astype("Int64")
)
both_top = (variants["training rank"] <= 50) & (
    variants["training rank (commonest)"] <= 50
)
print(
    f"Groups: {variants['commonest spelling'].nunique()}; spellings that "
    f"are not the commonest: {len(variants)}, on "
    f"{variants['lines'].sum():,} lines"
)
for kind, group in variants.groupby("kind", sort=False):
    print(f"  {kind}: {len(group)} spellings, {group['lines'].sum():,} lines")
print(
    "Spellings where both it and its commonest spelling are among the 50 "
    f"commonest types in training: {both_top.sum()}"
)
for row in variants[both_top].itertuples():
    print(f'  "{row.label}" for "{row._2}"')

# I put each label in quotes, so a space at either end shows.
labels = ["label", "commonest spelling"]
shown = variants.copy()
shown[labels] = shown[labels].apply(lambda column: '"' + column + '"')
shown

Groups: 161; spellings that are not the commonest: 176, on 3,729 lines
  spaces: 36 spellings, 438 lines
  case: 54 spellings, 1,432 lines
  dash: 5 spellings, 353 lines
  separator: 81 spellings, 1,506 lines
Spellings where both it and its commonest spelling are among the 50 commonest types in training: 2
  "TA Cost AUC ? Non SRN" for "TA Cost AUC – Non SRN"
  "TA Cost AUC / Programme" for "TA Cost AUC - Programme"


,label,commonest spelling,kind,lines,lines in commonest,training rank,training rank (commonest)
0,"""AUC""","""AUC """,spaces,140,334,86,41
1,"""IT SERVICES RUNNING COSTS""","""IT SERVICES RUNNING COSTS""",spaces,70,267,151,48
2,"""Business Rates ""","""Business Rates""",spaces,49,74,194,155
3,"""IT Consultancy ""","""IT Consultancy""",spaces,42,410,208,34
4,"""Service Charges""","""Service Charges """,spaces,29,58,299,196
5,"""RENT ""","""RENT""",spaces,15,817,367,21
6,""" L & P Partner: staffing assistance""","""L & P Partner: staffing assistance""",spaces,11,59,398,172
7,"""RENT ""","""RENT""",spaces,11,817,418,21
8,"""Rent - L&B ""","""Rent - L&B""",spaces,9,10,464,442
9,"""THEORY TEST VARIABLE CHARGES ""","""THEORY TEST VARIABLE CHARGES""",spaces,9,124,466,122


**Label variants.** In all, 161 Expense Type labels are spelled more than one way, and 176 spellings are not their label's commonest one, on 3,729 lines. Spaces make 36 of them, such as "RENT" with one, two or three spaces after it, and case makes 54, such as "RESEARCH" for "Research". A dash makes 5, such as the "Non SRN" capital label written with "?" for its dash in April, July and August 2023. A separator makes the other 81, such as "TA Cost AUC / Programme" in August 2022. In 2 pairs both spellings are among the 50 commonest types in training, so one account would count as two kept types. I propose merging every spelling into its commonest one before the kept types are chosen, and listing each merge in Task 3.

In [13]:
def truncation_pairs(
    frame,
    column="Expense Type",
    entity="Entity",
    month="month",
    min_length=10,
):
    """Return pairs of labels where the shorter may be the longer cut short.

    With case and every character that is not a letter or a digit ignored,
    the short label of a pair is at least min_length characters long and is
    the start of the long label, without matching it in full (a full match
    is a spelling variant, not a pair). Blank labels are left out. The rule
    also catches some distinct accounts, such as a general label and a more
    specific one, so every pair needs checking by eye. frame is not changed.

    Parameters
    ----------
    frame : pandas.DataFrame
        One row per line.
    column, entity, month : str
        The label, body and month columns.
    min_length : int, default 10
        The shortest folded label that can be the short side of a pair.

    Returns
    -------
    pandas.DataFrame
        One row per pair: short label, long label, lines (short), lines
        (long), characters (short) once trimmed, shared body (whether any
        body uses both labels) and months (short), the first and last month
        of the short label. Sorted by lines (short), most first.
    """
    lines = frame[frame[column].str.strip() != ""]
    counts = lines[column].value_counts()
    keys = counts.index.str.casefold().str.replace(
        r"[^a-z0-9]", "", regex=True
    )
    folded = pd.Series(keys, index=counts.index)
    folded = folded[folded != ""]
    bodies = {
        label: set(group.str.strip())
        for label, group in lines.groupby(column)[entity]
    }
    first = lines.groupby(column)[month].min()
    last = lines.groupby(column)[month].max()
    rows = []
    for short, short_key in folded.items():
        if len(short_key) < min_length:
            continue
        for long, long_key in folded.items():
            if long_key == short_key or not long_key.startswith(short_key):
                continue
            months = first[short]
            if last[short] != first[short]:
                months = f"{first[short]} to {last[short]}"
            rows.append(
                {
                    "short label": short,
                    "long label": long,
                    "lines (short)": counts[short],
                    "lines (long)": counts[long],
                    "characters (short)": len(short.strip()),
                    "shared body": bool(bodies[short] & bodies[long]),
                    "months (short)": months,
                }
            )
    out = pd.DataFrame(
        rows,
        columns=[
            "short label",
            "long label",
            "lines (short)",
            "lines (long)",
            "characters (short)",
            "shared body",
            "months (short)",
        ],
    )
    out = out.sort_values(
        ["lines (short)", "short label", "long label"],
        ascending=[False, True, True],
    )
    return out.reset_index(drop=True)

In [14]:
pairs = truncation_pairs(df)
shorts = pairs.drop_duplicates("short label")
print(
    f"Candidate pairs: {len(pairs)}; short labels: {len(shorts)}, on "
    f"{shorts['lines (short)'].sum():,} lines"
)
print(
    "Pairs whose short label is exactly 20 characters: "
    f"{(pairs['characters (short)'] == 20).sum()}; pairs that share a body: "
    f"{pairs['shared body'].sum()}"
)
longest = lines.groupby(lines["Entity"].str.strip())["Expense Type"].agg(
    lambda names: names.str.strip().str.len().max()
)
print(
    "Bodies whose longest Expense Type is 20 characters or fewer: "
    + ", ".join(f"{body} ({n})" for body, n in longest[longest <= 20].items())
)

labels = ["short label", "long label"]
shown = pairs.copy()
shown[labels] = shown[labels].apply(lambda column: '"' + column + '"')
shown

Candidate pairs: 241; short labels: 121, on 6,927 lines
Pairs whose short label is exactly 20 characters: 64; pairs that share a body: 132
Bodies whose longest Expense Type is 20 characters or fewer: Active Travel England (20), Department for Transport (20), East West Rail (18)


,short label,long label,lines (short),lines (long),characters (short),shared body,months (short)
0,"""Transport Consult""",""" Transport Consultancy""",1100,3,18,False,2022-04 to 2026-03
1,"""Transport Consult""","""Transport Consultancy""",1100,13,18,False,2022-04 to 2026-03
2,"""Mail Collection/Deli""","""Mail Collection/Delivery Costs""",611,550,20,True,2024-02 to 2026-03
3,"""Legal Consultancy""","""Legal Consultancy / Non Rec (DVLA only)""",608,1,17,True,2022-04 to 2026-03
4,"""Business Services""","""BUSINESS SERVICES DIRECTORATE""",553,3,17,False,2022-04 to 2026-03
5,"""IT Consultancy""",""" IT Consultancy - Other""",410,2,14,True,2022-04 to 2026-03
6,"""IT Consultancy""","""IT Consultancy - Non Rec""",410,12,14,True,2022-04 to 2026-03
7,"""IT Consultancy""","""IT Consultancy - Non Rec (DVLA only)""",410,17,14,True,2022-04 to 2026-03
8,"""IT Consultancy""","""IT Consultancy - Other""",410,12,14,True,2022-04 to 2026-03
9,"""IT Consultancy""","""IT Consultancy / Non Rec (DVLA only)""",410,9,14,True,2022-04 to 2026-03


**Truncation pairs.** The prefix rule finds 241 candidate pairs, with 121 short labels on 6,927 lines. Some are a label cut short, such as "Mail Collection/Deli" from February 2024 for "Mail Collection/Delivery Costs". Others are distinct accounts, such as "IT Consultancy" and "IT Consultancy - Non Rec", so I list every pair for a check by eye before any merge. The short label is exactly 20 characters in 64 pairs. The Department for Transport and Active Travel England never use a label longer than 20 characters, which suggests their system cuts labels at 20. So I also flag every label of exactly 20 characters in the anomaly list.

In [15]:
def labels_new_at_body(
    frame,
    body,
    prefix,
    since,
    column="Expense Type",
    entity="Entity",
    month="month",
    split="split",
):
    """Return the labels a body began to use from a given month.

    A label is listed when, once runs of spaces are closed up, it starts
    with prefix, and the body (the trimmed entity) uses it from since
    onwards but never before. Its first month and its lines in each split
    count every body's lines. Its status is "in training" when the training
    split holds at least one of its lines, and "first seen after training"
    otherwise. frame is not changed.

    Parameters
    ----------
    frame : pandas.DataFrame
        One row per line, with a split column.
    body : str
        The body, as the trimmed entity.
    prefix : str
        The start every listed label shares.
    since : str
        The month, "YYYY-MM", from which the body uses each label.
    column, entity, month, split : str
        The label, body, month and split columns.

    Returns
    -------
    pandas.DataFrame
        Indexed by label as published, in label order: first month,
        training, validation, test and status.
    """
    lines = frame[frame[column].str.strip() != ""]
    closed = lines[column].str.replace(r"\s+", " ", regex=True).str.strip()
    at_body = lines[
        (lines[entity].str.strip() == body) & closed.str.startswith(prefix)
    ]
    before = set(at_body.loc[at_body[month] < since, column])
    new = sorted(set(at_body.loc[at_body[month] >= since, column]) - before)
    chosen = lines[lines[column].isin(new)]
    splits = ["training", "validation", "test"]
    table = pd.DataFrame(
        {
            name: chosen[chosen[split] == name][column].value_counts()
            for name in splits
        },
        index=new,
    )
    table = table.fillna(0).astype(int)
    table.insert(0, "first month", chosen.groupby(column)[month].min())
    table["status"] = np.where(
        table["training"] > 0, "in training", "first seen after training"
    )
    return table

In [16]:
new_labels = labels_new_at_body(
    df, body="National Highways", prefix="TA Cost AUC", since="2025-04"
)
in_training = (new_labels["status"] == "in training").sum()
by_split = new_labels[["training", "validation", "test"]].sum()
print(
    f"{len(new_labels)} labels: {in_training} in training, "
    f"{len(new_labels) - in_training} first seen after training; lines: "
    f"training {by_split['training']:,}, validation "
    f"{by_split['validation']:,}, test {by_split['test']:,}"
)
test_lines = (lines["split"] == "test").sum()
print(f"Share of test lines: {by_split['test'] / test_lines:.1%}")
new_labels

15 labels: 12 in training, 3 first seen after training; lines: training 34, validation 185, test 2,091
Share of test lines: 12.8%


,first month,training,validation,test,status
TA Cost AUC,2026-03,0,0,19,first seen after training
TA Cost AUC - Ancillaries,2025-06,2,23,148,in training
TA Cost AUC - Capitalisable Pay,2026-01,0,0,1,first seen after training
TA Cost AUC - Design Services,2025-05,6,19,225,in training
TA Cost AUC - Drainage,2025-04,2,15,148,in training
TA Cost AUC - Geotechnics,2025-06,1,4,27,in training
TA Cost AUC - Lighting,2025-06,1,12,137,in training
TA Cost AUC - Pavements Flexible,2025-05,1,15,279,in training
TA Cost AUC - Pavements Rigid,2025-08,0,1,15,first seen after training
TA Cost AUC - Road Restraints,2025-06,5,18,149,in training


**The 2025 National Highways labels.** From April 2025 National Highways began to use 15 "TA Cost AUC" labels, such as "TA Cost AUC - Temporary Traffic Management". Of the 15, 12 appear in the training months, but rarely: 34 lines between them. The other 3 are novel classes, first seen after the training months: "TA Cost AUC - Pavements Rigid" from August 2025, "TA Cost AUC - Capitalisable Pay" in January 2026, and "TA  Cost AUC", published with two spaces, in March 2026. Validation holds 185 of their lines and test 2,091, which is 12.8% of the test lines. I keep them in the test months and report them as found.

In [17]:
def flag_anomalies(
    frame,
    variants,
    pairs,
    new_labels,
    published,
    cp850_files,
    drift_files,
    placeholders,
    hs2,
    hs2_labels,
    ewr,
    rare_body_lines=10,
    top_share=0.001,
    threshold=25_000,
):
    """Return one True or False column per kind of anomaly, for every row.

    A blank row (every published cell empty) is flagged as blank_row and
    nothing else. Every other kind looks only at lines with content. The
    kinds, in column order:

    - blank_row; cp850_file and header_drift: the line comes from one of
      cp850_files or drift_files;
    - spreadsheet_error: a cell holds "#N/A" or "N/A";
    - date_not_dd_mm_yyyy: the date is not written as dd/mm/yyyy;
    - date_outside_month: the date falls outside the file's month;
    - amount_unreadable, below_threshold, at_threshold: no amount, an
      amount under threshold, or exactly threshold;
    - very_large: the amount is in the top top_share of lines;
    - family_variant: Department Family is not its commonest spelling;
    - entity_spaces: Entity has spaces round it;
    - ewr: the trimmed Entity is one of the names in ewr;
    - rare_body: the trimmed Entity has fewer than rare_body_lines lines;
    - redacted: the Supplier contains "redact";
    - placeholder: the Supplier, in capitals, matches placeholders;
    - supplier_variant: the Supplier is not the commonest name among
      names that match once in capitals, with every character that is not
      a letter or a digit removed and a final LIMITED, LTD, PLC or LLP
      dropped (a tie goes to the name that sorts first);
    - exact_duplicate: every published cell repeats an earlier line;
    - near_duplicate: the line shares its Supplier, amount, date and
      Transaction No with another line that differs elsewhere;
    - split_transaction: its trimmed Transaction No carries more than one
      Expense Type in one body ("#", "N/A" and blank are not numbers);
    - no_transaction_number, no_item_text: "#", "N/A" or blank, and for
      Item Text also "##" and "#N/A";
    - label_20: the trimmed Expense Type is exactly 20 characters;
    - variant_spaces, variant_case, variant_dash, variant_separator: the
      Expense Type is a spelling of that kind in variants;
    - truncation: the Expense Type is a short label in pairs;
    - hs2_auc: an hs2 line whose label, spaces closed up, is in hs2_labels;
    - nh_2025: the Expense Type is in new_labels' index;
    - unseen_type: a validation or test line whose Expense Type never
      occurs in training.

    frame is not changed and no row is removed.

    Returns
    -------
    pandas.DataFrame
        One bool column per kind, in the order above, on frame's index.
    """
    trimmed = frame[published].apply(lambda column: column.str.strip())
    content = trimmed.ne("").any(axis=1)
    entity = trimmed["Entity"]
    label = frame["Expense Type"]
    closed = label.str.replace(r"\s+", " ", regex=True).str.strip()
    supplier = frame["Supplier"]
    number = trimmed["Transaction No"]
    amount = frame["amount"]

    families = frame.loc[content, "Department Family"].value_counts()
    bodies = entity[content].value_counts()
    cut = amount[content].quantile(1 - top_share)

    key = supplier.str.upper().str.replace(r"[^A-Z0-9]", "", regex=True)
    key = key.str.replace(r"(LIMITED|LTD|PLC|LLP)$", "", regex=True)
    names = pd.DataFrame({"key": key, "name": supplier})[content]
    names = names.value_counts().rename("lines").reset_index()
    names = names.sort_values(["lines", "name"], ascending=[False, True])
    commonest = names.groupby("key")["name"].first()
    shared = names.groupby("key")["name"].size() > 1
    shared_keys = shared.index[shared]

    rows = frame.loc[content, published + ["amount", "date"]]
    near_key = ["Supplier", "amount", "date", "Transaction No"]
    group = rows.groupby(near_key, dropna=False).ngroup()
    distinct = rows.assign(group=group).drop_duplicates(published)
    distinct = distinct.groupby("group").size()
    near = (group.map(group.value_counts()) > 1) & (group.map(distinct) > 1)

    numbered = content & ~number.isin(["", "#", "N/A"])
    types = pd.DataFrame(
        {"body": entity, "number": number, "label": label.str.strip()}
    )[numbered]
    types = types.groupby(["body", "number"])["label"].transform("nunique")

    in_training = content & (frame["split"] == "training")
    kind_of = variants.set_index("label")["kind"]

    flags = {
        "blank_row": ~content,
        "cp850_file": frame["source_file"].isin(cp850_files),
        "header_drift": frame["source_file"].isin(drift_files),
        "spreadsheet_error": trimmed.isin(["#N/A", "N/A"]).any(axis=1),
        "date_not_dd_mm_yyyy": ~frame["Date"].str.fullmatch(
            r"\d{2}/\d{2}/\d{4}"
        ),
        "date_outside_month": frame["date"].notna()
        & (frame["date"].dt.strftime("%Y-%m") != frame["month"]),
        "amount_unreadable": amount.isna(),
        "below_threshold": amount < threshold,
        "at_threshold": amount == threshold,
        "very_large": amount >= cut,
        "family_variant": frame["Department Family"] != families.index[0],
        "entity_spaces": frame["Entity"] != entity,
        "ewr": entity.isin(ewr),
        "rare_body": entity.map(bodies) < rare_body_lines,
        "redacted": supplier.str.contains("redact", case=False),
        "placeholder": supplier.str.upper().str.contains(placeholders),
        "supplier_variant": key.isin(shared_keys)
        & (supplier != key.map(commonest)),
        "exact_duplicate": frame.duplicated(published, keep="first"),
        "near_duplicate": near.reindex(frame.index, fill_value=False),
        "split_transaction": (types > 1).reindex(
            frame.index, fill_value=False
        ),
        "no_transaction_number": number.isin(["", "#", "N/A"]),
        "no_item_text": trimmed["Item Text"].isin(
            ["", "#", "##", "#N/A", "N/A"]
        ),
        "label_20": label.str.strip().str.len() == 20,
    }
    for kind in ("spaces", "case", "dash", "separator"):
        flags[f"variant_{kind}"] = label.isin(kind_of.index[kind_of == kind])
    flags["truncation"] = label.isin(pairs["short label"])
    flags["hs2_auc"] = (entity == hs2) & closed.isin(hs2_labels)
    flags["nh_2025"] = label.isin(new_labels.index)
    flags["unseen_type"] = (frame["split"] != "training") & ~label.isin(
        label[in_training]
    )
    out = pd.DataFrame(flags, index=frame.index).astype(bool)
    out.loc[~content] = False
    out["blank_row"] = ~content
    return out

In [18]:
def anomaly_table(frame, flags, kinds, amount="amount"):
    """Return the anomaly list: the lines and value of each kind.

    For each kind the list gives the lines flagged, their value, the
    published total with them, and the total without them. A line can be
    flagged for several kinds, so the rows do not add up to the file. No
    ruling is made here, so every ruling reads "for Paddy to rule". None of
    the inputs is changed.

    Parameters
    ----------
    frame : pandas.DataFrame
        One row per line, with an amount column.
    flags : pandas.DataFrame
        One bool column per kind, on frame's index.
    kinds : pandas.DataFrame
        Indexed by kind, in the same order as flags' columns: anomaly, the
        description, and proposed treatment.
    amount : str, default "amount"
        The amount column, in pounds.

    Returns
    -------
    pandas.DataFrame
        Numbered from 1, one row per kind: anomaly, lines, £ in these
        lines, total £ with them, total £ without them, proposed treatment
        and ruling.

    Raises
    ------
    ValueError
        If kinds does not list flags' columns in the same order.
    """
    if list(kinds.index) != list(flags.columns):
        raise ValueError("kinds must list the flag columns in the same order")
    total = frame[amount].sum()
    value = np.array(
        [frame.loc[flags[kind], amount].sum() for kind in flags.columns]
    )
    return pd.DataFrame(
        {
            "anomaly": kinds["anomaly"].to_numpy(),
            "lines": flags.sum().to_numpy(),
            "£ in these lines": value,
            "total £ with them": total,
            "total £ without them": total - value,
            "proposed treatment": kinds["proposed treatment"].to_numpy(),
            "ruling": "for Paddy to rule",
        },
        index=pd.RangeIndex(1, len(kinds) + 1, name="#"),
    )

In [19]:
# My proposed treatment for each kind: flag it (mark the lines and keep
# them), keep it (no action), or exclude it (leave the lines out of
# modelling, while the raw data keeps them). rare_body_lines is my own
# assumed limit, not a rule from the data.
rare_body_lines = 10
months_held = df["month"].nunique()
kinds = pd.DataFrame.from_records(
    [
        (
            "blank_row",
            "Blank row: every published cell empty",
            "exclude: no content",
        ),
        (
            "cp850_file",
            f"Line from one of the {len(cp850_files)} files that store £ as "
            "byte 0x9C (cp850)",
            "keep: the build reads them as cp850, so every amount is right",
        ),
        (
            "header_drift",
            f"Line from one of the {len(drift_files)} files that name a "
            "column differently",
            "keep: the build maps the names",
        ),
        (
            "spreadsheet_error",
            'Cell holding the spreadsheet error "#N/A" or "N/A"',
            "keep: treat the cell as missing",
        ),
        (
            "date_not_dd_mm_yyyy",
            "Date not written dd/mm/yyyy: dots, or spaces round it",
            "flag, keep: read as written",
        ),
        (
            "date_outside_month",
            "Line dated outside its file's month",
            "flag, keep",
        ),
        (
            "amount_unreadable",
            "Amount that cannot be read",
            "exclude: no amount",
        ),
        (
            "below_threshold",
            "Line under £25,000, zero or negative (a credit)",
            "flag, keep",
        ),
        (
            "at_threshold",
            "Line of exactly £25,000, the publishing threshold",
            "keep: at the threshold, not under it",
        ),
        (
            "very_large",
            "Very large line: the top 0.1% by amount",
            "flag, keep: the model takes the amount on a log scale",
        ),
        (
            "family_variant",
            "Department Family not in its commonest spelling",
            "keep: one department, so not a feature",
        ),
        (
            "entity_spaces",
            "Entity written with spaces round it",
            "keep: trim the name",
        ),
        (
            "ewr",
            '"EWR" and "East West Rail": one body under two names',
            "keep: merge into one body for results per body",
        ),
        (
            "rare_body",
            f"Body with fewer than an assumed {rare_body_lines} lines in "
            f"{months_held} months",
            "keep: too few lines to judge on its own",
        ),
        (
            "redacted",
            "Supplier redacted by DfT",
            "flag, keep: the supplier is unknown",
        ),
        (
            "placeholder",
            'Placeholder payee, such as "one time vendor"',
            "flag, keep",
        ),
        (
            "supplier_variant",
            "Supplier spelled differently from a commoner name for it "
            "(case, punctuation, Ltd or Limited)",
            "keep: the supplier feature reads parts of words, so near "
            "spellings share signal",
        ),
        (
            "exact_duplicate",
            "Exact duplicate: every published cell repeats an earlier line "
            "(the extra copies)",
            "exclude the extra copies from modelling; the raw data keeps "
            "them",
        ),
        (
            "near_duplicate",
            "Near-duplicate: same supplier, amount, date and transaction "
            "number, differing elsewhere (every line in the group)",
            "flag, keep: each may be a real posting",
        ),
        (
            "split_transaction",
            "Transaction number carrying more than one Expense Type in one "
            "body (every line)",
            "keep: the split is by month, never at random",
        ),
        (
            "no_transaction_number",
            'Transaction No missing ("#" or "N/A")',
            "keep: not a feature",
        ),
        (
            "no_item_text",
            'Item Text missing ("#", "##" or "#N/A")',
            "keep: treat as no text",
        ),
        (
            "label_20",
            "Expense Type exactly 20 characters long, likely cut off",
            "flag, keep",
        ),
        (
            "variant_spaces",
            "Expense Type differing from a commoner spelling in spaces only",
            "keep: merge into the commonest spelling",
        ),
        (
            "variant_case",
            "Expense Type differing from a commoner spelling in case",
            "keep: merge into the commonest spelling",
        ),
        (
            "variant_dash",
            "Expense Type differing from a commoner spelling in a dash, or "
            "a ? in its place",
            "keep: merge into the commonest spelling",
        ),
        (
            "variant_separator",
            "Expense Type differing from a commoner spelling in a separator, "
            "such as / for -",
            "keep: merge into the commonest spelling",
        ),
        (
            "truncation",
            "Expense Type that may be a longer label cut short (the short "
            "side of a pair)",
            "keep: merge the pairs confirmed by eye, and flag the rest",
        ),
        (
            "hs2_auc",
            'HS2 line labelled "AUC" or "AUC / Phase 1", where HS2 otherwise '
            'uses "AUC - Phase 1"',
            'keep: merge into "AUC - Phase 1" if confirmed by eye',
        ),
        (
            "nh_2025",
            f"The 2025 National Highways labels: the {len(new_labels)} "
            '"TA Cost AUC" labels '
            "first used from April 2025",
            "keep: report as found, as concept drift",
        ),
        (
            "unseen_type",
            "Validation or test line whose Expense Type never occurs in "
            "training",
            'keep: it falls into "Other"',
        ),
    ],
    columns=["kind", "anomaly", "proposed treatment"],
).set_index("kind")
print(f"Kinds of anomaly: {len(kinds)}")

Kinds of anomaly: 31


In [20]:
# A payee that is a placeholder for whoever was paid, not a supplier's name.
placeholders = (
    r"ONE TIME|ONE-TIME|ONE OFF|ONE-OFF|SUNDRY|SUNDRIES|MISCELLANEOUS|"
    r"VARIOUS|UNKNOWN|DUMMY"
)
flags = flag_anomalies(
    df,
    variants,
    pairs,
    new_labels,
    published=published,
    cp850_files=cp850_files,
    drift_files=drift_files,
    placeholders=placeholders,
    hs2="High Speed Two (HS2) Limited",
    hs2_labels=["AUC", "AUC / Phase 1"],
    ewr=["EWR", "East West Rail"],
    rare_body_lines=rare_body_lines,
    top_share=0.001,
    threshold=25_000,
)
anomalies = anomaly_table(df, flags, kinds)

copies = df[flags["exact_duplicate"]]
in_file = df.duplicated(published + ["source_file"]) & flags["exact_duplicate"]
print(f"Published total: £{df['amount'].sum():,.2f}")
print(
    f"Exact duplicates: {len(copies):,} extra copies, {in_file.sum():,} of "
    f"them in the same file as the line they repeat, in "
    f"{copies['source_file'].nunique()} files"
)
very_large = df.loc[flags["very_large"], "amount"]
print(f"Top 0.1% by amount: from £{very_large.min():,.2f}")
for name, group in df[flags["hs2_auc"]].groupby("Expense Type"):
    print(
        f'HS2 "{name}": {len(group):,} lines, {group["month"].min()} to '
        f'{group["month"].max()}'
    )
anomalies

Published total: £177,111,208,418.51
Exact duplicates: 329 extra copies, 329 of them in the same file as the line they repeat, in 47 files
Top 0.1% by amount: from £230,643,970.18
HS2 "AUC": 140 lines, 2023-07 to 2023-12
HS2 "AUC ": 334 lines, 2023-07 to 2023-12
HS2 "AUC / Phase 1": 61 lines, 2024-05 to 2024-05


,anomaly,lines,£ in these lines,total £ with them,total £ without them,proposed treatment,ruling
#,,,,,,,
1,Blank row: every published cell empty,657,0.00,"177,111,208,418.51","177,111,208,418.51",exclude: no content,for Paddy to rule
2,Line from one of the 3 files that store £ as byte 0x9C (cp850),8317,"12,296,156,491.49","177,111,208,418.51","164,815,051,927.02","keep: the build reads them as cp850, so every amount is right",for Paddy to rule
3,Line from one of the 5 files that name a column differently,13826,"18,263,369,361.27","177,111,208,418.51","158,847,839,057.24",keep: the build maps the names,for Paddy to rule
4,"Cell holding the spreadsheet error ""#N/A"" or ""N/A""",63,"15,424,730.99","177,111,208,418.51","177,095,783,687.52",keep: treat the cell as missing,for Paddy to rule
5,"Date not written dd/mm/yyyy: dots, or spaces round it",29,"1,909,226.26","177,111,208,418.51","177,109,299,192.25","flag, keep: read as written",for Paddy to rule
6,Line dated outside its file's month,24,"2,561,411.88","177,111,208,418.51","177,108,647,006.63","flag, keep",for Paddy to rule
7,Amount that cannot be read,0,0.00,"177,111,208,418.51","177,111,208,418.51",exclude: no amount,for Paddy to rule
8,"Line under £25,000, zero or negative (a credit)",0,0.00,"177,111,208,418.51","177,111,208,418.51","flag, keep",for Paddy to rule
9,"Line of exactly £25,000, the publishing threshold",171,"4,275,000.00","177,111,208,418.51","177,106,933,418.51","keep: at the threshold, not under it",for Paddy to rule


**What the anomaly list shows.** The 31 kinds are measured against a published total of £177,111,208,418.51. Two checks find nothing: every amount reads, and no line is under £25,000, so the file holds no credits.

- **Duplicates.** 329 lines repeat an earlier line in every published cell, each in the same file as the line it repeats, and are worth £52,280,103.56. I propose leaving these extra copies out of modelling, while the raw data keeps them. Another 300 lines share a supplier, amount, date and transaction number with a line that differs elsewhere, worth £497,459,805.53, and I propose to flag and keep them.
- **Payees.** DfT redacted the supplier on 3,563 lines worth £1,163,745,373.07, and 56 lines worth £8,307,591.37 go to placeholder payees.
- **Amounts.** The 126 lines in the top 0.1% by amount start at £230,643,970.18 and hold £51,557,542,687.81 between them.
- **Labels.** Spelling variants cover 3,729 lines, possible truncations 6,927 and labels of exactly 20 characters 22,261. HS2's relabelled AUC lines number 535. The 2025 National Highways labels hold 2,310 lines, and 295 validation or test lines carry a type never seen in training.

Every treatment in the list is a proposal until it is ruled on, and Task 3 applies the rulings.

## Modeling

## Evaluation

## Outputs

## Summary